# Deploy pipelines

Run this notebook after cloning the complete application. It loads the
account's shared settings, prepares sample data only when no dataset is
configured, and publishes the training and inference pipelines.

On later runs it validates and reuses the configured inputs. Existing
namespace and data choices are retained. Partial or inaccessible inputs
stop deployment rather than triggering replacement data.

The operator first creates `wsf-mlops-setup` using
`operations/create_setup_secret.ipynb`. Existing notebooks 01 and 02
remain the detailed preparation and publication views.

The shared data seed is **42**. To reproduce the same input bytes, use the same
metadata, model targets, generation settings, code and dependency versions.
Compare the three published SHA-256 values. Reusing data does not apply a new
seed: select `generate` explicitly if you intentionally change generation settings.


## Choose deployment options

- Use the current AWS identity and region; the secret supplies platform settings.
- Use `auto` to generate only when no dataset is configured, or `reuse` to require existing data.
- Select `generate` only when you deliberately want a new sample.
- Keep shared-setting refresh off unless you intend to replace conflicting local platform values.


In [ ]:
from pathlib import Path
import json
import sys
import boto3

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src").is_dir())
SOURCE = ROOT / "notebooks/source/setup"
sys.path.insert(0, str(SOURCE))
import deploy_pipelines as deployment
assert Path(deployment.__file__).resolve() == (SOURCE / "deploy_pipelines.py").resolve(), (
    "Restart the kernel: the deployment helper came from another checkout."
)

SECRET_NAME = "wsf-mlops-setup"
REGION = None  # Use the AWS region configured in this environment.
DATA_MODE = "auto"
REFRESH_SHARED = False
DATA_SEED = 42  # Keep this identical to notebook 01 and other workspaces.
PREPARATION = {"sample_scale": 100, "numeric_zero_fraction": 0.10, "split_seed": DATA_SEED}
PUBLICATION = {}  # Existing defaults; e.g. {"model_ids": ["usecase1"], "cache_steps": True}.
session = boto3.Session(region_name=REGION)


## Deploy and return the result

- Check config state and fill missing shared values from the secret.
- Generate missing data or validate the exact configured files/S3 objects.
- Verify uploads and upsert pipeline definitions using the existing publication flow.
- Return the result below; detailed output and any failure are saved under `.runs/setup/`.


In [ ]:
result = deployment.deploy_pipelines(
    ROOT, session,
    secret_name=SECRET_NAME,
    refresh_shared=REFRESH_SHARED,
    data_mode=DATA_MODE,
    preparation_options=PREPARATION,
    publication_options=PUBLICATION,
)


## Read the result

- Show elapsed time and five actual training rows; `<missing>` marks blank CSV cells.
- For generated data, show the seed, split sizes and missing-cell counts.
- Print the exact published S3 paths and hashes for comparing inputs between workspaces.
- Keep full diagnostics in the saved report/log; continue with notebook 05 to execute pipelines.


In [ ]:
import pandas as pd

print(f"Completed in {result['elapsed_seconds']:.1f}s | Data: {result['data_action']}")
if "preparation" in result:
    prepared = result["preparation"]
    print(f"Seed: {prepared['seed']} | Generation: {prepared['elapsed_seconds']:.1f}s")
    print("Rows: " + " | ".join(f"{kind}: {count:,}" for kind, count in prepared["rows"].items()))
    print("Missing cells: " + " | ".join(f"{kind}: {count:,}" for kind, count in prepared["missing_cells"].items()))
    if any(prepared["missing_cells"].values()):
        print("Source missing values were preserved; column counts are in the saved preparation report.")
if result["preview"]:
    print("\nTraining sample — first 5 rows, first 6 columns (<missing> = blank CSV cell):")
    print(pd.DataFrame(result["preview"]).replace("", "<missing>").to_string(index=False, max_colwidth=24))
print("\nPublished data:")
for kind, key in (("training", "csv_input"), ("testing", "test_csv_input"), ("inference", "inference_csv_input")):
    if result["inputs"].get(key):
        print(f"  {kind}: {result['inputs'][key]}")
        if kind in result["sha256"]:
            print(f"    SHA-256: {result['sha256'][kind]}")
print("\nPipelines: " + " | ".join(item["name"] for item in result["pipelines"].values()))
print("Details:", result["report_path"])
